# US Valuation 통합 조회 v1

FCFF DCF · RIM · Relative Valuation 3개 모형의 **미국 주식** DB 저장 결과를 통합 조회하고
upside(현재 주가 대비 상승여력) 내림차순 순위로 추출 → 엑셀(`C:\us valuation results`)로
저장하는 노트북입니다. `korea_valuation_query_v5.ipynb` 의 미국판 포팅입니다.

| 함수 | 설명 |
|---|---|
| `get_valuation_dates()` | 모형별 valuation 수행 날짜 + 종목 수 조회 |
| `get_fcff_ranking()` | FCFF DCF upside 순위 추출 (`us_fcff_dcf_valuation`) |
| `get_rim_ranking()` | RIM upside 순위 추출 (`us_rim_valuation`) |
| `get_relative_ranking()` | 상대가치 upside 순위 추출 — PER/PBR/PSR 선택 (`us_relative_valuation`) |
| `get_combined_ranking()` | 3개 모형 통합 비교 순위 (DB 재조회) |
| `merge_upside_rankings()` | 이미 추출된 순위표 2~3개 병합 → 동일가중 평균 upside 순위 |
| `get_revenue_growth()` | 매출 성장률(4Q/8Q) 단독 조회 (`us_revenue_forecast_data`) |
| `build_growth_table()` | 전 종목 성장률 테이블 생성·캐시 |
| `diagnose_growth()` | 특정 종목 성장률 NaN 원인 진단 |

**공통 파라미터**
- `n=50` : 상위 N개 (rank_range 미지정 시)
- `rank_range=(100, 150)` : 순위 구간 출력 (지정 시 n 무시)
- `dates=["2026-06-01", "2026-06-02"]` : 평가일 리스트. `None` → 최신 평가일 자동.
  배치가 2일 이상 걸쳐 저장된 경우를 위한 리스트 입력. 여러 날 중복 종목은 **최신 측정일 기준** 1행만 사용.
- `save=True` : 엑셀 저장 여부 (기본 True). `file_format="csv"` 도 가능.
- `max_upside=None` : 데이터 품질용 upside 상한 필터 (예: 300 → +300% 초과 제외)
- `growth_model="Ensemble"` : 매출 성장률 산출 모델. `"SARIMA"`/`"ETS"`/`"Theta"` 지정 가능,
  `None` → 성장률 칼럼 생략

**상대가치 지표 안내 (⚠ PCR 관련)**
- 미국 상대가치 노트북(`Relative_Valuation_v8`)은 **PER / PBR / PSR** 3종의 적정주가·upside를
  DB에 저장합니다. **PCR(주가현금흐름비율)은 산출·저장되지 않으므로** 본 조회 코드에서도
  PSR이 세 번째 지표 자리를 대신합니다. (한국판 v5와 동일한 구성)

**매출 성장률 정의**
- `매출성장률_4Q(%)` = Σ(예측 1~4분기 매출) ÷ Σ(예측 직전 4개 실적 분기 매출) − 1
- `매출성장률_8Q(%)` = Σ(예측 1~8분기 매출) ÷ Σ(예측 직전 8개 실적 분기 매출) − 1
- 종목별로 **가장 최근 forecast_date** 버전 사용. 실적/예측 분기 수 부족 시 NaN.

**저장 파일명**: `{Valuation_method}_valuation_{수행날짜}_{출력날짜}.xlsx`
(예: `FCFF_valuation_2026-06-01_to_2026-06-02_2026-06-06.xlsx`)

In [1]:
# ── Cell 1 · 경로 자동 감지 (기존 US valuation 노트북과 동일) ───
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "DATA")):
            if cand not in sys.path:
                sys.path.insert(0, cand)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다. _CANDIDATE_ROOTS 를 환경에 맞게 수정하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")

[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [2]:
# ── Cell 2 · Import & 설정 상수 ─────────────────────────────────
import os
import time
from datetime import datetime
from typing import Optional, List, Tuple, Union, Dict

import numpy as np
import pandas as pd
import pymysql
import requests
from IPython.display import display

from DATA.config import get_db_info, get_engine

def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)

# ══════════════════════════════════════════════════════════════
#  설정 — 여기만 수정하세요
# ══════════════════════════════════════════════════════════════

# 3개 valuation 결과 테이블 (각 노트북의 TABLE_RESULT 와 동일)
TABLE_FCFF = "us_fcff_dcf_valuation"     # FCFF_DCF_Valuation_v12
TABLE_RIM  = "us_rim_valuation"          # RIM_Valuation_v11
TABLE_REL  = "us_relative_valuation"     # Relative_Valuation_v8

# 매출 예측 결과 (long: ticker·item·data_type·model·date·value·forecast_date)
TABLE_FORECAST = "us_revenue_forecast_data"
GROWTH_MODELS  = {"ENSEMBLE": "Ensemble", "SARIMA": "SARIMA",
                  "ETS": "ETS", "THETA": "Theta"}

# 엑셀 저장 폴더 (요구사항: C 드라이브 'us valuation results')
OUTPUT_DIR = r"C:\us valuation results"

# 상대가치 지표명 ↔ DB 컬럼 suffix 매핑
#  ※ 미국 상대가치 노트북은 PER/PBR/PSR 3종을 저장합니다.
#    PCR(주가현금흐름비율)은 DB에 없으므로 PSR 이 그 자리를 대신합니다.
REL_METRICS_ALL = ["PER", "PBR", "PSR"]

# 종목명/섹터 조회용 FMP profile API (기존 노트북과 동일 키)
FMP_API_KEY = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
FMP_BASE    = "https://financialmodelingprep.com/api/v3"
FMP_SLEEP   = 0.25
DEFAULT_PORT = 3307

db_info = get_db_info()
engine  = get_engine(db_info)   # (연결 테스트용 — 조회는 아래 cursor 패턴 사용)

# ── DB 연결: cursor.execute → fetchall → DataFrame 패턴 ─────────
#    (pd.read_sql + pymysql 비호환 방지 — 기존 US 노트북과 동일)
def get_conn():
    return pymysql.connect(
        host       = db_info["host"],
        port       = int(db_info.get("port", DEFAULT_PORT)),
        user       = db_info["user"],
        password   = db_info["password"],
        db         = db_info.get("database", "investar"),
        charset    = "utf8mb4",
        autocommit = False,
        cursorclass= pymysql.cursors.DictCursor,
    )

def _query(sql: str, params=None) -> pd.DataFrame:
    """SELECT 전용 헬퍼: cursor → fetchall → DataFrame."""
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            rows = cur.fetchall()
    finally:
        conn.close()
    return pd.DataFrame(rows)

try:
    _query("SELECT 1 AS ok")
    log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
except Exception as e:
    log("DB", f"연결 실패: {e}")

print(f"[설정] FCFF={TABLE_FCFF}  RIM={TABLE_RIM}  REL={TABLE_REL}")
print(f"[설정] 저장 폴더 = {OUTPUT_DIR}")

[10:55:56][DB] 연결 성공 host=192.168.0.230 port=3307
[설정] FCFF=us_fcff_dcf_valuation  RIM=us_rim_valuation  REL=us_relative_valuation
[설정] 저장 폴더 = C:\us valuation results


In [3]:
# ── Cell 3 · 종목명/섹터 매핑 (FMP profile, 실패해도 진행) ──────
#  미국 DB에는 회사명이 저장돼 있지 않아 FMP /profile 로 조회합니다.
#  - 출력 대상 티커만 조회(상위 N개 수준) → API 부담 최소화
#  - 세션 내 캐시(PROFILE_CACHE) → 같은 티커 재조회 없음
#  - batch(콤마 결합) 우선, 실패 시 단건 fallback, 그것도 실패하면 공란 진행

PROFILE_CACHE: Dict[str, Dict] = {}

def _norm_tk(t) -> str:
    return str(t).strip().upper()

def _parse_profile(d: dict) -> Dict:
    return {
        "name":    d.get("companyName") or "",
        "sector":  d.get("sector") or "",
        "industry":d.get("industry") or "",
        "mkt_cap": d.get("mktCap"),
        "exchange":d.get("exchangeShortName") or "",
    }

def fetch_profiles(tickers, batch_size: int = 25, verbose: bool = False) -> int:
    """캐시에 없는 티커만 FMP profile 조회. 반환: 신규 조회 성공 건수."""
    miss = [t for t in dict.fromkeys(map(_norm_tk, tickers)) if t not in PROFILE_CACHE]
    if not miss:
        return 0
    n_ok = 0
    for i in range(0, len(miss), batch_size):
        batch = miss[i:i + batch_size]
        got = set()
        # 1) batch 호출
        try:
            r = requests.get(f"{FMP_BASE}/profile/{','.join(batch)}",
                             params={"apikey": FMP_API_KEY}, timeout=15)
            data = r.json()
            if isinstance(data, list):
                for d in data:
                    sym = _norm_tk(d.get("symbol", ""))
                    if sym:
                        PROFILE_CACHE[sym] = _parse_profile(d)
                        got.add(sym); n_ok += 1
        except Exception as e:
            if verbose:
                log("NAME", f"[WARN] batch 조회 실패: {str(e)[:60]}")
        # 2) 누락분 단건 fallback
        for t in batch:
            if t in got:
                continue
            try:
                r = requests.get(f"{FMP_BASE}/profile/{t}",
                                 params={"apikey": FMP_API_KEY}, timeout=10)
                data = r.json()
                if isinstance(data, list) and data:
                    PROFILE_CACHE[t] = _parse_profile(data[0]); n_ok += 1
                else:
                    PROFILE_CACHE[t] = {"name": "", "sector": "", "industry": "",
                                        "mkt_cap": None, "exchange": ""}
            except Exception:
                PROFILE_CACHE[t] = {"name": "", "sector": "", "industry": "",
                                    "mkt_cap": None, "exchange": ""}
            time.sleep(FMP_SLEEP)
        time.sleep(FMP_SLEEP)
    return n_ok

def get_name_sector(ticker: str) -> Dict:
    t = _norm_tk(ticker)
    if t not in PROFILE_CACHE:
        fetch_profiles([t])
    return PROFILE_CACHE.get(t, {"name": "", "sector": "", "industry": "",
                                 "mkt_cap": None, "exchange": ""})

def _attach_profile(df: pd.DataFrame,
                    add_sector: bool = True,
                    add_mktcap: bool = True) -> pd.DataFrame:
    """ticker 다음 위치에 종목명(+섹터) 삽입, 끝에 시가총액($B) 추가.

    ※ 시가총액은 FMP profile '현재' 기준 값으로, 측정일 시점과 다를 수 있습니다.
    """
    if df.empty or "ticker" not in df.columns:
        return df
    fetch_profiles(df["ticker"].tolist())
    prof = df["ticker"].map(lambda t: PROFILE_CACHE.get(_norm_tk(t), {}))
    pos = df.columns.get_loc("ticker") + 1
    df.insert(pos, "종목명", prof.map(lambda p: p.get("name", "")))
    if add_sector and "섹터" not in df.columns:
        df.insert(pos + 1, "섹터", prof.map(lambda p: p.get("sector", "")))
    if add_mktcap and "시가총액($B)" not in df.columns:
        mc = prof.map(lambda p: p.get("mkt_cap"))
        df["시가총액($B)"] = (pd.to_numeric(mc, errors="coerce") / 1e9).round(2)
    return df

log("NAME", "FMP profile 매핑 준비 완료 (출력 시 필요한 티커만 조회·캐시)")

[10:55:56][NAME] FMP profile 매핑 준비 완료 (출력 시 필요한 티커만 조회·캐시)


In [4]:
# ── Cell 4 · 공통 유틸 (날짜 조회/정규화, 순위 슬라이스, 파일 저장) ──

_MODEL_TABLE = {"FCFF": TABLE_FCFF, "RIM": TABLE_RIM, "Relative": TABLE_REL}


def get_valuation_dates(model: str = "all", verbose: bool = True) -> pd.DataFrame:
    """[요구사항 12] valuation 수행 날짜 조회.

    Parameters
    ----------
    model : "fcff" | "rim" | "relative" | "all"

    Returns
    -------
    DataFrame [model, run_date, n_tickers, n_rows, first_saved, last_saved] (최신순)
    """
    key = str(model).strip().lower()
    targets = (list(_MODEL_TABLE.items()) if key == "all"
               else [(m, t) for m, t in _MODEL_TABLE.items() if m.lower() == key])
    if not targets:
        raise ValueError(f"model='{model}' 인식 불가. 'fcff'/'rim'/'relative'/'all' 중 선택.")

    frames = []
    for mname, tbl in targets:
        sql = f"""
            SELECT '{mname}' AS model, `date` AS run_date,
                   COUNT(DISTINCT ticker) AS n_tickers, COUNT(*) AS n_rows,
                   MIN(created_at) AS first_saved, MAX(created_at) AS last_saved
            FROM `{tbl}`
            GROUP BY `date`
            ORDER BY `date` DESC
        """
        try:
            df = _query(sql)
            if not df.empty:
                frames.append(df)
        except Exception as e:
            log("DATES", f"[WARN] {mname}({tbl}) 조회 실패: {e}")
    if not frames:
        out = pd.DataFrame(columns=["model", "run_date", "n_tickers", "n_rows",
                                    "first_saved", "last_saved"])
    else:
        out = pd.concat(frames, ignore_index=True)
        out["run_date"] = pd.to_datetime(out["run_date"]).dt.strftime("%Y-%m-%d")
        out = out.sort_values(["model", "run_date"],
                              ascending=[True, False]).reset_index(drop=True)
    if verbose and not out.empty:
        display(out)
    return out


def _resolve_dates(dates, table: str, model_name: str) -> List[str]:
    """입력 dates(None/str/list) → 해당 테이블에 실제 존재하는 날짜 리스트.

    - None / [] → 최신 평가일 1일 자동
    - 존재하지 않는 날짜는 경고 후 제외
    """
    avail = _query(f"SELECT DISTINCT `date` FROM `{table}` ORDER BY `date` DESC")
    if avail.empty:
        raise ValueError(f"[{model_name}] {table} 에 저장된 결과가 없습니다.")
    avail_set = set(pd.to_datetime(avail["date"]).dt.strftime("%Y-%m-%d"))

    if dates is None or (isinstance(dates, (list, tuple)) and len(dates) == 0):
        latest = max(avail_set)
        log(model_name, f"날짜 미지정 → 최신 평가일 사용: {latest}")
        return [latest]

    if isinstance(dates, str):
        dates = [dates]
    norm = [pd.to_datetime(d).strftime("%Y-%m-%d") for d in dates]
    missing = sorted(set(norm) - avail_set)
    if missing:
        log(model_name, f"⚠️  측정 기록 없는 날짜 (무시): {missing}")
    valid = sorted(set(norm) & avail_set)
    if not valid:
        raise ValueError(f"[{model_name}] 입력한 날짜에 측정 기록이 없습니다. "
                         f"get_valuation_dates('{model_name.lower()}') 로 확인하세요.")
    return valid


def _slice_rank(df: pd.DataFrame, n: int,
                rank_range: Optional[Tuple[int, int]]) -> pd.DataFrame:
    """upside 내림차순 정렬 상태에서 rank 부여 → 상위 n 또는 rank_range 구간."""
    df = df.reset_index(drop=True)
    df.insert(0, "rank", df.index + 1)
    if rank_range is not None:
        lo, hi = int(rank_range[0]), int(rank_range[1])
        if lo > hi:
            lo, hi = hi, lo
        out = df[(df["rank"] >= lo) & (df["rank"] <= hi)]
        if out.empty:
            log("RANK", f"⚠️  rank_range=({lo},{hi}) 구간에 종목 없음 (전체 {len(df)}개)")
        return out
    return df.head(int(n))


def _save_output(df: pd.DataFrame, method: str, run_dates: List[str],
                 save: bool = True, file_format: str = "xlsx") -> Optional[str]:
    """OUTPUT_DIR 에 {Method}_valuation_{수행날짜}_{출력날짜}.xlsx 형식 저장.

    [요구사항 10] save 기본값 True, 폴더 자동 생성.
    """
    if not save:
        return None
    if df.empty:
        log("SAVE", "결과가 비어 있어 저장 생략")
        return None
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    run_tag = (run_dates[0] if len(run_dates) == 1
               else f"{min(run_dates)}_to_{max(run_dates)}")
    today = datetime.now().strftime("%Y-%m-%d")
    ext = "csv" if str(file_format).lower() == "csv" else "xlsx"
    fname = f"{method}_valuation_{run_tag}_{today}.{ext}"
    path = os.path.join(OUTPUT_DIR, fname)
    if ext == "csv":
        df.to_csv(path, index=False, encoding="utf-8-sig")
    else:
        df.to_excel(path, index=False)
    log("SAVE", f"저장 완료: {path}  ({len(df)}행)")
    return path

In [5]:
# ── Cell 5 · 매출 성장률 (4Q/8Q): 예측 전량 로드 → merge 방식 ────
#  us_revenue_forecast_data (long: ticker·item·data_type·model·date·value·forecast_date)
#  - forecast: 종목별 '가장 최근 forecast_date' 버전만 사용
#  - actual  : (ticker,date)별 가장 최근 forecast_date 행만 사용 (v12 결정성 패턴)
#  - 4Q = 향후 4분기 예측합 ÷ 직전 4분기 실적합 − 1,  8Q 동일
#  ※ 미국 데이터는 전부 FMP 단일 출처 → 한국판의 단위(10^k) 자동보정은 생략

_GROWTH_COLS  = ["매출성장률_4Q(%)", "매출성장률_8Q(%)"]
_GROWTH_CACHE: Dict[str, pd.DataFrame] = {}
_MODEL_PRIORITY = ["Ensemble", "SARIMA", "ETS", "Theta"]


def _load_forecast_all() -> pd.DataFrame:
    """종목별 최신 forecast_date 의 forecast 행 전량 로드."""
    sql = f"""
        SELECT a.ticker, a.model, a.date, a.value
        FROM `{TABLE_FORECAST}` a
        INNER JOIN (
            SELECT ticker, MAX(forecast_date) AS max_fd
            FROM `{TABLE_FORECAST}`
            WHERE item='sale' AND data_type='forecast'
            GROUP BY ticker
        ) b ON a.ticker = b.ticker AND a.forecast_date = b.max_fd
        WHERE a.item='sale' AND a.data_type='forecast'
    """
    df = _query(sql)
    if df.empty:
        return df
    df["ticker"] = df["ticker"].map(_norm_tk)
    df["date"]   = pd.to_datetime(df["date"])
    df["value"]  = pd.to_numeric(df["value"], errors="coerce")
    return df.dropna(subset=["value"])


def _load_actual_all() -> pd.DataFrame:
    """(ticker,date)별 최신 forecast_date 의 actual 행 전량 로드."""
    sql = f"""
        SELECT a.ticker, a.date, a.value
        FROM `{TABLE_FORECAST}` a
        INNER JOIN (
            SELECT ticker, `date`, MAX(forecast_date) AS max_fd
            FROM `{TABLE_FORECAST}`
            WHERE item='sale' AND data_type='actual'
            GROUP BY ticker, `date`
        ) b ON a.ticker = b.ticker AND a.date = b.date AND a.forecast_date = b.max_fd
        WHERE a.item='sale' AND a.data_type='actual'
    """
    df = _query(sql)
    if df.empty:
        return df
    df["ticker"] = df["ticker"].map(_norm_tk)
    df["date"]   = pd.to_datetime(df["date"])
    df["value"]  = pd.to_numeric(df["value"], errors="coerce")
    return (df.dropna(subset=["value"])
              .sort_values(["ticker", "date"])
              .drop_duplicates(["ticker", "date"], keep="last"))


def build_growth_table(model: str = "Ensemble", refresh: bool = False) -> pd.DataFrame:
    """전 종목 매출성장률(4Q/8Q) 테이블 생성 + 세션 캐시.

    model : "Ensemble"|"SARIMA"|"ETS"|"Theta"
            해당 모델 예측이 없는 종목은 우선순위(Ensemble→SARIMA→ETS→Theta) fallback.
    refresh : True → 캐시 무시하고 DB 재조회
    """
    key = GROWTH_MODELS.get(str(model).strip().upper())
    if key is None:
        raise ValueError(f"model='{model}' 인식 불가. {list(GROWTH_MODELS.values())} 중 선택.")
    if (not refresh) and key in _GROWTH_CACHE:
        return _GROWTH_CACHE[key]

    fc  = _load_forecast_all()
    act = _load_actual_all()
    log("GROWTH", f"예측 {len(fc):,}행 / 실적 {len(act):,}행 로드 "
                  f"(예측 보유 종목 {fc['ticker'].nunique() if not fc.empty else 0:,}개)")
    if fc.empty or act.empty:
        out = pd.DataFrame(columns=["ticker", "사용모델"] + _GROWTH_COLS)
        _GROWTH_CACHE[key] = out
        return out

    act_g = {t: g for t, g in act.groupby("ticker")}
    recs = []
    order = [key] + [m for m in _MODEL_PRIORITY if m != key]
    for tk, g in fc.groupby("ticker"):
        avail = g["model"].unique().tolist()
        used = next((m for m in order if m in avail), None)
        if used is None:
            continue
        f = (g[g["model"] == used].sort_values("date")
             .drop_duplicates("date", keep="last"))
        a = act_g.get(tk)
        if a is None or f.empty:
            continue
        fd0 = f["date"].iloc[0]
        past = a[a["date"] < fd0].sort_values("date")
        g4 = g8 = np.nan
        if len(f) >= 4 and len(past) >= 4:
            s_f4, s_a4 = f["value"].iloc[:4].sum(), past["value"].iloc[-4:].sum()
            if s_a4 > 0:
                g4 = (s_f4 / s_a4 - 1) * 100
        if len(f) >= 8 and len(past) >= 8:
            s_f8, s_a8 = f["value"].iloc[:8].sum(), past["value"].iloc[-8:].sum()
            if s_a8 > 0:
                g8 = (s_f8 / s_a8 - 1) * 100
        recs.append({"ticker": tk, "사용모델": used,
                     _GROWTH_COLS[0]: round(g4, 1) if pd.notna(g4) else np.nan,
                     _GROWTH_COLS[1]: round(g8, 1) if pd.notna(g8) else np.nan})

    out = pd.DataFrame(recs)
    n4 = int(out[_GROWTH_COLS[0]].notna().sum()) if not out.empty else 0
    n8 = int(out[_GROWTH_COLS[1]].notna().sum()) if not out.empty else 0
    log("GROWTH", f"성장률 테이블 {len(out):,}종목 (4Q 산출 {n4:,} / 8Q 산출 {n8:,}) — 캐시 저장")
    _GROWTH_CACHE[key] = out
    return out


def _merge_growth(out: pd.DataFrame, growth_model: Optional[str]) -> pd.DataFrame:
    """순위표의 Upside 칼럼 바로 뒤에 매출성장률 4Q/8Q 칼럼 삽입."""
    if growth_model is None or out.empty or "ticker" not in out.columns:
        return out
    try:
        tbl = build_growth_table(growth_model)
    except Exception as e:
        log("GROWTH", f"[WARN] 성장률 병합 생략: {e}")
        return out
    if tbl.empty:
        return out
    out = out.copy()
    out["_tk"] = out["ticker"].map(_norm_tk)
    out = out.merge(tbl[["ticker"] + _GROWTH_COLS].rename(columns={"ticker": "_tk"}),
                    on="_tk", how="left").drop(columns="_tk")
    # Upside 칼럼 바로 뒤로 이동
    up_idx = next((i for i, c in enumerate(out.columns)
                   if str(c).startswith("Upside")), None)
    if up_idx is not None:
        cols = [c for c in out.columns if c not in _GROWTH_COLS]
        insert_at = cols.index(out.columns[up_idx]) + 1 if out.columns[up_idx] in cols else len(cols)
        cols = cols[:insert_at] + _GROWTH_COLS + cols[insert_at:]
        out = out[cols]
    return out


def get_revenue_growth(tickers: Union[None, str, List[str]] = None,
                       model: str = "Ensemble") -> pd.DataFrame:
    """매출 성장률(4Q/8Q) 단독 조회. tickers=None → 전 종목."""
    tbl = build_growth_table(model)
    if tickers is None:
        return tbl.sort_values(_GROWTH_COLS[0], ascending=False).reset_index(drop=True)
    if isinstance(tickers, str):
        tickers = [tickers]
    want = [_norm_tk(t) for t in tickers]
    out = tbl[tbl["ticker"].isin(want)].reset_index(drop=True)
    miss = sorted(set(want) - set(out["ticker"]))
    if miss:
        log("GROWTH", f"⚠️  성장률 미산출 종목: {miss} → diagnose_growth() 로 원인 확인")
    return out


def diagnose_growth(ticker: str, model: str = "Ensemble"):
    """특정 종목 성장률 NaN 원인 단계별 진단."""
    tk = _norm_tk(ticker)
    fc  = _load_forecast_all()
    act = _load_actual_all()
    f = fc[fc["ticker"] == tk] if not fc.empty else pd.DataFrame()
    a = act[act["ticker"] == tk] if not act.empty else pd.DataFrame()
    print(f"[진단] {tk}")
    print(f"  1) forecast 행: {len(f)}  (모델: {sorted(f['model'].unique()) if not f.empty else '없음'})")
    print(f"  2) actual  행: {len(a)}")
    if f.empty:
        print("  → forecast 없음: us_revenue_forecast_notebook 먼저 실행 필요"); return
    if a.empty:
        print("  → actual 없음: 예측 원천 데이터 적재 필요"); return
    key = GROWTH_MODELS.get(str(model).strip().upper(), "Ensemble")
    order = [key] + [m for m in _MODEL_PRIORITY if m != key]
    used = next((m for m in order if m in f["model"].unique()), None)
    ff = f[f["model"] == used].sort_values("date")
    fd0 = ff["date"].iloc[0]
    past = a[a["date"] < fd0]
    print(f"  3) 사용 모델: {used}  |  예측 분기 수: {len(ff)} (4Q 필요 ≥4, 8Q 필요 ≥8)")
    print(f"  4) 예측 시작 {fd0.date()} 이전 실적 분기 수: {len(past)} (4Q 필요 ≥4, 8Q 필요 ≥8)")
    if len(ff) < 4 or len(past) < 4:
        print("  → 분기 수 부족이 NaN 원인입니다.")
    else:
        print("  → 분기 수는 충분 — value 합계 0/음수 여부 확인:")
        print(f"     예측4Q합={ff['value'].iloc[:4].sum():.3e}  "
              f"실적4Q합={past['value'].iloc[-4:].sum():.3e}")

In [6]:
# ── Cell 6 · FCFF DCF 순위 추출 ─────────────────────────────────
#  us_fcff_dcf_valuation 은 (ticker, date, quarter) 분기 행 구조.
#  TP/현재가/upside/WACC 등 요약값은 분기 행마다 동일 → (ticker,date) 집계 후
#  종목별 최신 평가일(동일자면 최신 created_at) 1행만 사용.
#  ※ 스키마 참고: 이 테이블에는 Re/Rd/beta/moat 칼럼이 없어 WACC(discount_rate)만
#    표기합니다 (FCFF v12 Cell 9 패치 적용 후 저장분부터는 beta/re/rd 확장 가능).

def get_fcff_ranking(n: int = 50,
                     rank_range: Optional[Tuple[int, int]] = None,
                     dates: Union[None, str, List[str]] = None,
                     growth_model: Optional[str] = "Ensemble",
                     save: bool = True,
                     file_format: str = "xlsx",
                     max_upside: Optional[float] = None) -> pd.DataFrame:
    """FCFF DCF 결과를 upside 내림차순 순위로 추출.

    Parameters
    ----------
    n          : 상위 N개 (rank_range 지정 시 무시)                       [요구사항 5·7]
    rank_range : (100, 150) 처럼 순위 구간 출력                            [요구사항 8]
    dates      : ["2026-06-01","2026-06-02"] 평가일 리스트. None → 최신일. [요구사항 13]
                 여러 날 중복 종목은 최신 측정일 행만 사용.                 [요구사항 14]
    save       : True → 'C:\\us valuation results' 에 엑셀 저장 (기본 True) [요구사항 10]
    file_format: "xlsx"(기본) | "csv"
    max_upside : upside 상한 필터(%). 예: 300 → +300% 초과 이상치 제외
    growth_model : 매출성장률 모델. "Ensemble"(기본)|"SARIMA"|"ETS"|"Theta"|None
    """
    run_dates = _resolve_dates(dates, TABLE_FCFF, "FCFF")
    ph = ", ".join(["%s"] * len(run_dates))

    sql = f"""
        SELECT * FROM (
            SELECT t.*,
                   ROW_NUMBER() OVER (
                       PARTITION BY ticker ORDER BY measured_date DESC, last_saved DESC
                   ) AS rn
            FROM (
                SELECT ticker, `date` AS measured_date,
                       MAX(target_price)      AS target_price,
                       MAX(current_price)     AS current_price,
                       MAX(upside_pct)        AS upside_pct,
                       MAX(discount_rate)     AS wacc,
                       MAX(g_terminal)        AS g_terminal,
                       MAX(reinvestment_rate) AS reinvestment_rate,
                       AVG(roic)              AS roic_avg,
                       MAX(enterprise_value)  AS enterprise_value,
                       MAX(equity_value)      AS equity_value,
                       MAX(net_debt)          AS net_debt,
                       MAX(shares)            AS shares,
                       MAX(created_at)        AS last_saved
                FROM `{TABLE_FCFF}`
                WHERE `date` IN ({ph})
                  AND target_price IS NOT NULL
                  AND current_price > 0
                  AND upside_pct IS NOT NULL
                GROUP BY ticker, `date`
            ) t
        ) z WHERE rn = 1
    """
    raw = _query(sql, run_dates)
    if raw.empty:
        log("FCFF", "조건에 맞는 결과 없음")
        return raw
    raw = raw.drop(columns=["rn"])
    for c in ["target_price", "current_price", "upside_pct", "wacc", "g_terminal",
              "reinvestment_rate", "roic_avg", "enterprise_value",
              "equity_value", "net_debt", "shares"]:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

    if max_upside is not None:
        n_drop = int((raw["upside_pct"] > max_upside).sum())
        raw = raw[raw["upside_pct"] <= max_upside]
        if n_drop:
            log("FCFF", f"upside > {max_upside}% 이상치 {n_drop}개 제외")

    raw["measured_date"] = pd.to_datetime(raw["measured_date"]).dt.strftime("%Y-%m-%d")
    raw = raw.sort_values("upside_pct", ascending=False)   # [요구사항 6]

    out = pd.DataFrame({
        "ticker":        raw["ticker"].map(_norm_tk),
        "측정일":         raw["measured_date"],
        "TP($)":         raw["target_price"].round(2),
        "현재가($)":      raw["current_price"].round(2),
        "Upside(%)":     raw["upside_pct"].round(1),
        "WACC(%)":       (raw["wacc"] * 100).round(2),
        "g_term(%)":     (raw["g_terminal"] * 100).round(2),
        "재투자율(%)":     (raw["reinvestment_rate"] * 100).round(1),
        "ROIC평균(%)":    (raw["roic_avg"] * 100).round(1),
        "EV($B)":        (raw["enterprise_value"] / 1e9).round(2),
        "지분가치($B)":    (raw["equity_value"] / 1e9).round(2),
        "순부채($B)":     (raw["net_debt"] / 1e9).round(2),
        "주식수(M)":      (raw["shares"] / 1e6).round(1),
    })
    out = _slice_rank(out, n, rank_range)
    out = _attach_profile(out)                       # 종목명·섹터·시총   [요구사항 15]
    out = _merge_growth(out, growth_model)

    print(f"\n{'='*95}")
    print(f"  [FCFF DCF] upside 순위  |  평가일 {run_dates}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*95}")
    display(out)
    _save_output(out, "FCFF", run_dates, save=save, file_format=file_format)
    return out

In [7]:
# ── Cell 7 · RIM 순위 추출 ──────────────────────────────────────
#  us_rim_valuation 은 (ticker, date, year_label) 연도 행 구조.
#  TP/upside/Re/moat 등 요약값은 행마다 동일 저장 → (ticker,date) 집계 후
#  종목별 최신 평가일 1행만 사용.

def get_rim_ranking(n: int = 50,
                    rank_range: Optional[Tuple[int, int]] = None,
                    dates: Union[None, str, List[str]] = None,
                    growth_model: Optional[str] = "Ensemble",
                    save: bool = True,
                    file_format: str = "xlsx",
                    max_upside: Optional[float] = None) -> pd.DataFrame:
    """RIM(잔여이익모형) 결과를 upside 내림차순 순위로 추출.

    파라미터는 get_fcff_ranking 과 동일.
    """
    run_dates = _resolve_dates(dates, TABLE_RIM, "RIM")
    ph = ", ".join(["%s"] * len(run_dates))

    sql = f"""
        SELECT * FROM (
            SELECT t.*,
                   ROW_NUMBER() OVER (
                       PARTITION BY ticker ORDER BY measured_date DESC, last_saved DESC
                   ) AS rn
            FROM (
                SELECT ticker, `date` AS measured_date,
                       MAX(target_price)    AS target_price,
                       MAX(current_price)   AS current_price,
                       MAX(upside_pct)      AS upside_pct,
                       MAX(intrinsic_value) AS intrinsic_value,
                       MAX(re)              AS re,
                       MAX(g_terminal)      AS g_terminal,
                       MAX(rho)             AS rho,
                       MAX(n_phase2)        AS n_phase2,
                       MAX(moat_label)      AS moat_label,
                       MAX(created_at)      AS last_saved
                FROM `{TABLE_RIM}`
                WHERE `date` IN ({ph})
                  AND target_price IS NOT NULL
                  AND current_price > 0
                  AND upside_pct IS NOT NULL
                GROUP BY ticker, `date`
            ) t
        ) z WHERE rn = 1
    """
    raw = _query(sql, run_dates)
    if raw.empty:
        log("RIM", "조건에 맞는 결과 없음")
        return raw
    raw = raw.drop(columns=["rn"])
    for c in ["target_price", "current_price", "upside_pct", "intrinsic_value",
              "re", "g_terminal", "rho", "n_phase2"]:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

    if max_upside is not None:
        n_drop = int((raw["upside_pct"] > max_upside).sum())
        raw = raw[raw["upside_pct"] <= max_upside]
        if n_drop:
            log("RIM", f"upside > {max_upside}% 이상치 {n_drop}개 제외")

    raw["measured_date"] = pd.to_datetime(raw["measured_date"]).dt.strftime("%Y-%m-%d")
    raw = raw.sort_values("upside_pct", ascending=False)

    out = pd.DataFrame({
        "ticker":         raw["ticker"].map(_norm_tk),
        "측정일":          raw["measured_date"],
        "TP($)":          raw["target_price"].round(2),
        "현재가($)":       raw["current_price"].round(2),
        "Upside(%)":      raw["upside_pct"].round(1),
        "내재가치($B)":     (raw["intrinsic_value"] / 1e9).round(2),
        "Re(%)":          (raw["re"] * 100).round(2),
        "g_term(%)":      (raw["g_terminal"] * 100).round(2),
        "RI지속계수ω":     raw["rho"].round(3),
        "Phase2연수":      raw["n_phase2"],
        "Moat":           raw["moat_label"],
    })
    out = _slice_rank(out, n, rank_range)
    out = _attach_profile(out)
    out = _merge_growth(out, growth_model)

    print(f"\n{'='*95}")
    print(f"  [RIM] upside 순위  |  평가일 {run_dates}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*95}")
    display(out)
    _save_output(out, "RIM", run_dates, save=save, file_format=file_format)
    return out

In [8]:
# ── Cell 8 · Relative Valuation 순위 추출 (PER/PBR/PSR 선택) ────
#  us_relative_valuation 은 (ticker, date) 1행 구조.
#  ⚠ PCR 은 상대가치 노트북에서 산출하지 않아 DB에 없습니다 — PSR 이 대체.

def get_relative_ranking(n: int = 50,
                         rank_range: Optional[Tuple[int, int]] = None,
                         dates: Union[None, str, List[str]] = None,
                         metrics: Union[str, List[str]] = "ALL",
                         growth_model: Optional[str] = "Ensemble",
                         save: bool = True,
                         file_format: str = "xlsx",
                         include_excluded: bool = False,
                         max_upside: Optional[float] = None) -> pd.DataFrame:
    """상대가치 결과를 upside 내림차순 순위로 추출.

    Parameters
    ----------
    metrics : "PER" | "PBR" | "PSR" | ["PER","PBR"] | "ALL"            [요구사항 9]
        - 1개 지정 → 해당 지표 upside 기준 정렬
        - 2개 이상/ALL → 선택 지표 upside 평균 기준 정렬
    include_excluded : True → 금융 등 제외 섹터(is_excluded=1)도 포함
    나머지 파라미터는 get_fcff_ranking 과 동일.
    """
    # ── 지표 정규화 ──
    if isinstance(metrics, str):
        metrics = REL_METRICS_ALL if metrics.strip().upper() == "ALL" else [metrics]
    mets = [m.strip().upper() for m in metrics]
    bad = [m for m in mets if m not in REL_METRICS_ALL]
    if bad:
        raise ValueError(
            f"지원하지 않는 지표 {bad}. 사용 가능: {REL_METRICS_ALL} "
            f"(PCR 은 상대가치 노트북에서 산출하지 않아 DB에 없습니다 — PSR 사용 권장)")

    run_dates = _resolve_dates(dates, TABLE_REL, "Relative")
    ph = ", ".join(["%s"] * len(run_dates))
    excl_clause = "" if include_excluded else "AND is_excluded = 0"

    sql = f"""
        SELECT * FROM (
            SELECT
                ticker, `date` AS measured_date, sector, is_excluded,
                current_price,
                tp_per, tp_pbr, tp_psr, tp_avg,
                upside_per, upside_pbr, upside_psr, upside_avg,
                per_theory, pbr_theory, psr_theory,
                actual_per, actual_pbr, actual_psr,
                roe_y2, re_mid, g_est, eps_y2, bps_y2, sps_y2, beta_ensemble,
                ROW_NUMBER() OVER (
                    PARTITION BY ticker ORDER BY `date` DESC, id DESC
                ) AS rn
            FROM `{TABLE_REL}`
            WHERE `date` IN ({ph})
              AND current_price > 0
              {excl_clause}
        ) t WHERE rn = 1
    """
    raw = _query(sql, run_dates)
    if raw.empty:
        log("REL", "조건에 맞는 결과 없음")
        return raw
    raw = raw.drop(columns=["rn"])
    num_cols = [c for c in raw.columns if c not in ("ticker", "measured_date", "sector")]
    for c in num_cols:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

    # ── 정렬 기준: 선택 지표 upside (1개=그 지표, 복수=평균) ──
    up_cols = [f"upside_{m.lower()}" for m in mets]
    raw["upside_sel"] = raw[up_cols].mean(axis=1, skipna=True)
    raw = raw[raw["upside_sel"].notna()]
    if max_upside is not None:
        n_drop = int((raw["upside_sel"] > max_upside).sum())
        raw = raw[raw["upside_sel"] <= max_upside]
        if n_drop:
            log("REL", f"upside > {max_upside}% 이상치 {n_drop}개 제외")

    raw["measured_date"] = pd.to_datetime(raw["measured_date"]).dt.strftime("%Y-%m-%d")
    raw = raw.sort_values("upside_sel", ascending=False)

    sel_label = "+".join(mets)
    cols = {
        "ticker":      raw["ticker"].map(_norm_tk),
        "측정일":       raw["measured_date"],
        "섹터":         raw["sector"],
        "현재가($)":    raw["current_price"].round(2),
        f"Upside_{sel_label}(%)": raw["upside_sel"].round(1),
    }
    for m in mets:  # 지표별 적정가/upside/이론·실제 멀티플
        lm = m.lower()
        cols[f"TP_{m}($)"]     = raw[f"tp_{lm}"].round(2)
        cols[f"Upside_{m}(%)"] = raw[f"upside_{lm}"].round(1)
        cols[f"{m}_이론"]       = raw[f"{lm}_theory"].round(2)
        cols[f"{m}_실제"]       = raw[f"actual_{lm}"].round(2)
    cols.update({
        "ROE_y2(%)":   (raw["roe_y2"] * 100).round(2),
        "Re_mid(%)":   (raw["re_mid"] * 100).round(2),
        "g_est(%)":    (raw["g_est"] * 100).round(2),
        "EPS_y2($)":   raw["eps_y2"].round(2),
        "BPS_y2($)":   raw["bps_y2"].round(2),
        "SPS_y2($)":   raw["sps_y2"].round(2),
        "β_ensemble":  raw["beta_ensemble"].round(3),
    })
    if include_excluded:
        cols["제외섹터"] = raw["is_excluded"]
    out = pd.DataFrame(cols)
    out = _slice_rank(out, n, rank_range)
    out = _attach_profile(out)     # 섹터는 DB 값이 이미 있어 종목명·시총만 추가됨
    out = _merge_growth(out, growth_model)

    print(f"\n{'='*95}")
    print(f"  [Relative {sel_label}] upside 순위  |  평가일 {run_dates}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*95}")
    display(out)
    _save_output(out, f"Relative_{sel_label}", run_dates,
                 save=save, file_format=file_format)
    return out

In [9]:
# ── Cell 9 · 3개 모형 통합 비교 순위 (DB 재조회) ────────────────

def _fetch_model_upside(table: str, model_name: str, dates) -> pd.DataFrame:
    """모형별 (ticker → 최신 측정일 TP/현재가/upside) 공통 추출."""
    run_dates = _resolve_dates(dates, table, model_name)
    ph = ", ".join(["%s"] * len(run_dates))

    if table == TABLE_REL:
        # 상대가치는 (ticker,date) 1행 → 그대로 ROW_NUMBER
        sql = f"""
            SELECT ticker, measured_date, tp, cp, upside FROM (
                SELECT ticker, `date` AS measured_date,
                       tp_avg AS tp, current_price AS cp, upside_avg AS upside,
                       ROW_NUMBER() OVER (
                           PARTITION BY ticker ORDER BY `date` DESC, id DESC
                       ) AS rn
                FROM `{table}`
                WHERE `date` IN ({ph})
                  AND current_price > 0
                  AND is_excluded = 0 AND tp_avg IS NOT NULL
            ) t WHERE rn = 1
        """
    else:
        # FCFF/RIM 은 분기·연도 다행 → (ticker,date) 집계 후 ROW_NUMBER
        sql = f"""
            SELECT ticker, measured_date, tp, cp, upside FROM (
                SELECT t.*,
                       ROW_NUMBER() OVER (
                           PARTITION BY ticker
                           ORDER BY measured_date DESC, last_saved DESC
                       ) AS rn
                FROM (
                    SELECT ticker, `date` AS measured_date,
                           MAX(target_price)  AS tp,
                           MAX(current_price) AS cp,
                           MAX(upside_pct)    AS upside,
                           MAX(created_at)    AS last_saved
                    FROM `{table}`
                    WHERE `date` IN ({ph})
                      AND current_price > 0
                      AND target_price IS NOT NULL AND upside_pct IS NOT NULL
                    GROUP BY ticker, `date`
                ) t
            ) z WHERE rn = 1
        """
    df = _query(sql, run_dates)
    if not df.empty:
        df["ticker"] = df["ticker"].map(_norm_tk)
        for c in ["tp", "cp", "upside"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["measured_date"] = pd.to_datetime(df["measured_date"]).dt.strftime("%Y-%m-%d")
    df.attrs["run_dates"] = run_dates
    return df


def get_combined_ranking(n: int = 50,
                         rank_range: Optional[Tuple[int, int]] = None,
                         dates: Union[None, str, List[str]] = None,
                         growth_model: Optional[str] = "Ensemble",
                         save: bool = True,
                         file_format: str = "xlsx",
                         min_models: int = 1,
                         max_upside: Optional[float] = None) -> pd.DataFrame:
    """FCFF · RIM · Relative(tp_avg/upside_avg) 3개 모형 upside 를 종목별 병합한 통합 순위.

    - 정렬 기준: 가용 모형 upside 의 단순평균 (Upside_평균)
    - min_models : 최소 몇 개 모형에서 평가된 종목만 포함 (기본 1, 보수적으로 3)
    - dates=None → 모형별 각각의 최신 평가일 사용
    """
    parts = {}
    for label, tbl, mname in [("FCFF", TABLE_FCFF, "FCFF"),
                              ("RIM", TABLE_RIM, "RIM"),
                              ("REL", TABLE_REL, "Relative")]:
        try:
            df = _fetch_model_upside(tbl, mname, dates)
            if not df.empty:
                parts[label] = df
        except ValueError as e:
            log("COMBINED", f"[WARN] {label} 건너뜀: {e}")

    if not parts:
        log("COMBINED", "사용 가능한 모형 결과가 없습니다.")
        return pd.DataFrame()

    merged = None
    all_run_dates = []
    for label, df in parts.items():
        all_run_dates += df.attrs.get("run_dates", [])
        sub = df.rename(columns={
            "tp": f"TP_{label}($)", "upside": f"Upside_{label}(%)",
            "cp": f"_cp_{label}", "measured_date": f"_dt_{label}"})
        merged = sub if merged is None else merged.merge(sub, on="ticker", how="outer")

    up_cols = [c for c in merged.columns if c.startswith("Upside_")]
    cp_cols = [c for c in merged.columns if c.startswith("_cp_")]
    dt_cols = [c for c in merged.columns if c.startswith("_dt_")]

    merged["평가모형수"] = merged[up_cols].notna().sum(axis=1)
    merged = merged[merged["평가모형수"] >= int(min_models)]
    merged["Upside_평균(%)"] = merged[up_cols].mean(axis=1, skipna=True)
    if max_upside is not None:
        merged = merged[merged["Upside_평균(%)"] <= max_upside]
    merged["현재가($)"]  = merged[cp_cols].bfill(axis=1).iloc[:, 0]
    merged["최신측정일"] = merged[dt_cols].max(axis=1)

    merged = merged.sort_values("Upside_평균(%)", ascending=False)

    ordered = (["ticker", "최신측정일", "현재가($)", "Upside_평균(%)", "평가모형수"]
               + sorted(up_cols)
               + sorted(c for c in merged.columns if c.startswith("TP_")))
    out = merged[ordered].copy()
    out["현재가($)"] = out["현재가($)"].round(2)
    for c in out.columns:
        if c.startswith("Upside"):
            out[c] = out[c].round(1)
        elif c.startswith("TP_"):
            out[c] = out[c].round(2)
    out = _slice_rank(out, n, rank_range)
    out = _attach_profile(out)
    out = _merge_growth(out, growth_model)

    run_dates = sorted(set(all_run_dates))
    print(f"\n{'='*100}")
    print(f"  [통합 Combined] FCFF·RIM·Relative upside 평균 순위  |  "
          f"평가일 {run_dates}  |  min_models={min_models}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*100}")
    display(out)
    _save_output(out, "Combined", run_dates, save=save, file_format=file_format)
    return out

In [10]:
# ── Cell 10 · 추출된 순위표 병합 → 통합 upside 분석 ─────────────
#
#  get_fcff_ranking / get_rim_ranking / get_relative_ranking 이 "반환한"
#  DataFrame 2~3개를 ticker 기준 merge → 모형별 Upside + 동일가중 평균 Upside.
#  DB 재조회 없이 화면에 띄워 둔 순위표(필터·rank_range 적용 결과)를 그대로 병합.
#  ※ get_combined_ranking 과의 차이: combined 는 DB에서 다시 읽음.

_MERGE_G_MAP = {"매출성장률_4Q(%)": "_g4", "매출성장률_8Q(%)": "_g8"}


def _detect_model_label(df: pd.DataFrame) -> str:
    """순위표 DataFrame 의 출처 모형 자동 판별 → 'FCFF' / 'RIM' / 'REL'."""
    cols = set(map(str, df.columns))
    if {"WACC(%)", "재투자율(%)", "EV($B)"} & cols:
        return "FCFF"
    if {"RI지속계수ω", "내재가치($B)", "Phase2연수"} & cols:
        return "RIM"
    if "섹터" in cols and any(str(c).startswith("Upside_") for c in cols):
        return "REL"
    raise ValueError(
        "모형 판별 불가 — get_fcff_ranking / get_rim_ranking / "
        "get_relative_ranking 이 반환한 DataFrame 을 그대로 입력하세요. "
        f"(columns 앞부분: {list(df.columns)[:8]})")


def _pick_upside_col(df: pd.DataFrame, label: str) -> str:
    """모형별 대표 upside 칼럼명 결정.

    FCFF/RIM → 'Upside(%)'
    REL      → 집계 upside (예: 'Upside_PER+PBR+PSR(%)') = 첫 'Upside_' 칼럼
    """
    if label in ("FCFF", "RIM"):
        if "Upside(%)" in df.columns:
            return "Upside(%)"
    else:
        for c in df.columns:
            s = str(c)
            if s.startswith("Upside_") and s.endswith("(%)"):
                return s
    raise ValueError(f"[{label}] upside 칼럼을 찾을 수 없습니다. "
                     f"(columns 앞부분: {list(df.columns)[:8]})")


def merge_upside_rankings(dfs: Union[List[pd.DataFrame], Dict[str, pd.DataFrame]],
                          n: Optional[int] = None,
                          rank_range: Optional[Tuple[int, int]] = None,
                          min_models: int = 1,
                          max_upside: Optional[float] = None,
                          save: bool = True,
                          file_format: str = "xlsx") -> pd.DataFrame:
    """추출된 순위표 2~3개를 ticker 기준 병합한 통합 upside 순위.

    Parameters
    ----------
    dfs : list 또는 dict (2~3개)
        - list → 칼럼 구성으로 모형 자동 판별
        - dict → {"FCFF": df1, "RIM": df2, "REL": df3} 라벨 직접 지정
    n / rank_range : 출력 구간. 기본 None → 전체 출력.
    min_models : 최소 평가 모형 수 (기본 1 = 합집합, 입력 개수와 같게 주면 교집합)
    max_upside : Upside_평균(%) 상한 필터 (이상치 제외용)
    save : True → OUTPUT_DIR 에 'Merged_{모형들}_valuation_...' 저장
    """
    # ── 1) 입력 정규화 + 모형 판별 ──
    if isinstance(dfs, dict):
        items = [(str(k).strip().upper(), v) for k, v in dfs.items()]
    else:
        items = [(_detect_model_label(df), df) for df in dfs]
    if not 2 <= len(items) <= 3:
        raise ValueError(f"DataFrame 은 2~3개 입력하세요 (현재 {len(items)}개).")
    labels = [lb for lb, _ in items]
    if len(set(labels)) != len(labels):
        raise ValueError(f"같은 모형이 중복 입력되었습니다: {labels}")

    # ── 2) 모형별 핵심 칼럼 추출 → ticker 기준 outer merge ──
    merged: Optional[pd.DataFrame] = None
    run_dates: set = set()
    for label, df in items:
        if df is None or df.empty:
            raise ValueError(f"[{label}] 입력 DataFrame 이 비어 있습니다.")
        if "ticker" not in df.columns:
            raise ValueError(f"[{label}] 'ticker' 칼럼이 없습니다.")
        d = df.copy()
        up_col = _pick_upside_col(d, label)

        d["_tk"] = d["ticker"].map(_norm_tk)
        n_dup = int(d["_tk"].duplicated().sum())
        if n_dup:
            log("MERGE", f"⚠️ [{label}] ticker 중복 {n_dup}건 → 첫 행(상위 순위) 사용")
        d = d.drop_duplicates("_tk", keep="first")

        part = pd.DataFrame({"_tk": d["_tk"].values})
        part[f"Upside_{label}(%)"] = pd.to_numeric(d[up_col], errors="coerce").values
        if "종목명" in d.columns:
            part[f"_nm_{label}"] = d["종목명"].values
        if "섹터" in d.columns:
            part[f"_sec_{label}"] = d["섹터"].values
        if "Moat" in d.columns:
            part[f"_moat_{label}"] = d["Moat"].values
        for gcol, tag in _MERGE_G_MAP.items():
            if gcol in d.columns:
                part[f"{tag}_{label}"] = pd.to_numeric(d[gcol], errors="coerce").values
        if "측정일" in d.columns:
            part[f"_dt_{label}"] = d["측정일"].astype(str).values
            run_dates |= set(part[f"_dt_{label}"].dropna())

        log("MERGE", f"[{label}] {len(part):,}종목 | upside='{up_col}' | "
                     f"Moat={'O' if 'Moat' in d.columns else 'X'} | "
                     f"성장률={'O' if _GROWTH_COLS[0] in d.columns else 'X'}")
        merged = part if merged is None else merged.merge(part, on="_tk", how="outer")

    # ── 3) coalesce (먼저 입력된 df 우선) ──
    def _coalesce(prefix: str) -> pd.Series:
        cols = [f"{prefix}_{lb}" for lb in labels if f"{prefix}_{lb}" in merged.columns]
        if not cols:
            return pd.Series(np.nan, index=merged.index)
        s = merged[cols[0]]
        for c in cols[1:]:
            s = s.combine_first(merged[c])
        return s

    up_cols = [f"Upside_{lb}(%)" for lb in labels]
    out = pd.DataFrame({"ticker": merged["_tk"]})
    out["종목명"] = _coalesce("_nm").fillna("")
    out["섹터"]   = _coalesce("_sec").fillna("")
    dt_cols = [c for c in merged.columns if c.startswith("_dt_")]
    if dt_cols:   # outer merge NaN(float) + 문자열 혼합 → datetime 변환 후 max
        _dt = merged[dt_cols].apply(lambda s: pd.to_datetime(s, errors="coerce"))
        out["최신측정일"] = _dt.max(axis=1).dt.strftime("%Y-%m-%d").fillna("")
    else:
        out["최신측정일"] = ""
    out["Moat"] = _coalesce("_moat")
    for c in up_cols:
        out[c] = merged[c].round(1)
    out["평가모형수"]      = merged[up_cols].notna().sum(axis=1)
    out["Upside_평균(%)"] = merged[up_cols].mean(axis=1, skipna=True).round(1)
    out["매출성장률_4Q(%)"] = _coalesce("_g4")
    out["매출성장률_8Q(%)"] = _coalesce("_g8")

    # ── 4) 필터 → 평균 upside 내림차순 정렬 ──
    n_all = len(out)
    out = out[out["평가모형수"] >= int(min_models)]
    if max_upside is not None:
        n_drop = int((out["Upside_평균(%)"] > max_upside).sum())
        out = out[out["Upside_평균(%)"] <= max_upside]
        if n_drop:
            log("MERGE", f"Upside_평균 > {max_upside}% 이상치 {n_drop}개 제외")
    out = out.sort_values("Upside_평균(%)", ascending=False, na_position="last")

    ordered = (["ticker", "종목명", "섹터", "최신측정일", "Moat",
                "Upside_평균(%)", "평가모형수"] + up_cols + _GROWTH_COLS)
    out = out[ordered]
    out = _slice_rank(out, n if n is not None else len(out), rank_range)

    # ── 5) 출력 + 저장 ──
    tag = "+".join(labels)
    log("MERGE", f"병합 완료: 합집합 {n_all:,}종목 → "
                 f"min_models≥{min_models} 적용 후 {len(out):,}종목 출력 "
                 f"(평균 = {tag} 동일가중)")
    print(f"\n{'='*100}")
    print(f"  [Merged {tag}] 동일가중 평균 upside 순위  |  min_models={min_models}  |  "
          f"{'rank ' + str(rank_range) if rank_range else ('전체' if n is None else 'TOP ' + str(n))}")
    print(f"{'='*100}")
    display(out)
    rd = sorted(run_dates) if run_dates else [datetime.now().strftime("%Y-%m-%d")]
    _save_output(out, f"Merged_{tag}", rd, save=save, file_format=file_format)
    return out

In [11]:
# ── Cell 11 · 0단계: valuation 수행 날짜부터 확인 ───────────────
dates_df = get_valuation_dates("all")

,model,run_date,n_tickers,n_rows,first_saved,last_saved
0,FCFF,2026-06-30,1616,12928,2026-06-30 21:00:14,2026-07-01 06:18:47
1,FCFF,2026-06-21,7,56,2026-06-21 11:42:33,2026-06-21 11:56:04
2,FCFF,2026-06-15,1445,11560,2026-06-15 00:25:45,2026-06-15 08:38:53
3,FCFF,2026-06-14,10,80,2026-06-14 23:22:09,2026-06-14 23:59:24
4,FCFF,2026-06-12,835,6680,2026-06-12 17:11:43,2026-06-12 21:36:03
5,FCFF,2026-06-07,835,6680,2026-06-07 00:35:29,2026-06-07 05:04:43
6,FCFF,2026-06-06,1,8,2026-06-06 23:37:42,2026-06-06 23:37:42
7,FCFF,2026-05-30,1,8,2026-05-30 14:15:56,2026-05-30 14:15:56
8,FCFF,2026-05-29,12,96,2026-05-29 16:51:35,2026-05-29 17:13:40
9,FCFF,2026-05-07,4,32,2026-05-07 09:20:00,2026-05-07 19:20:14


## 사용 예시

```python
# 0) 어떤 날짜에 valuation 이 수행됐는지 먼저 확인              [요구사항 12]
get_valuation_dates("all")        # 3개 모형 전체
get_valuation_dates("fcff")       # FCFF 만

# 1) FCFF 상위 30개, 최신 평가일, 엑셀 저장(기본 True)
fcff_top = get_fcff_ranking(n=30)

# 2) RIM 100~150위 구간, 특정 날짜 2일 (중복 종목은 최신 측정일 우선)  [요구사항 8·13·14]
rim_mid = get_rim_ranking(rank_range=(100, 150),
                          dates=["2026-06-01", "2026-06-02"])

# 3) 상대가치 — PER 단독 / PBR+PSR 조합 / 전체 3종              [요구사항 9]
rel_per = get_relative_ranking(n=50, metrics="PER")
rel_mix = get_relative_ranking(n=50, metrics=["PBR", "PSR"])
rel_all = get_relative_ranking(n=50, metrics="ALL", save=False)   # 저장 생략
# ※ PCR 은 DB에 산출·저장되지 않아 PSR 이 세 번째 지표입니다.

# 4) 통합(DB 재조회) — 3개 모형 모두 평가된 종목만, upside 평균 상위 50
combo = get_combined_ranking(n=50, min_models=3)

# 5) 매출 성장률 — 순위표에 자동 병합 (기본 Ensemble)
fcff   = get_fcff_ranking(n=30)                           # Ensemble 성장률 포함
fcff_s = get_fcff_ranking(n=30, growth_model="SARIMA")    # SARIMA 성장률
fcff_x = get_fcff_ranking(n=30, growth_model=None)        # 성장률 칼럼 생략

# 6) 성장률 NaN 원인 진단 / 테이블 직접 확인
diagnose_growth("NVDA")
tbl = build_growth_table("Ensemble")                  # 전 종목 성장률 (캐시)
tbl = build_growth_table("Ensemble", refresh=True)    # DB 재조회 갱신

# 7) 순수 매출 성장률 단독 조회
g_all  = get_revenue_growth()                         # 전체 종목, Ensemble
g_some = get_revenue_growth(["NVDA", "AAPL"], model="Theta")

# 8) 추출된 순위표 병합 — 동일가중 평균 upside (DB 재조회 없음)
fcff_all = get_fcff_ranking(rank_range=(0, 2000), dates=["2026-06-01"], save=False)
rel_all  = get_relative_ranking(rank_range=(0, 2000), metrics="ALL", save=False)
rim_all  = get_rim_ranking(rank_range=(0, 2000), save=False)

merged = merge_upside_rankings([fcff_all, rel_all, rim_all])           # 자동 판별, 합집합
merged = merge_upside_rankings([fcff_all, rim_all])                    # 2개만도 가능
merged = merge_upside_rankings([fcff_all, rel_all, rim_all],
                               min_models=3)                           # 교집합만
merged = merge_upside_rankings({"FCFF": fcff_all, "REL": rel_all},
                               max_upside=300, save=False)             # 라벨 직접 지정

# 9) 데이터 품질 필터 — upside +300% 초과 이상치 제외 (g≈WACC 발산 케이스 등)
fcff_clean = get_fcff_ranking(n=100, max_upside=300)
```

저장 파일명 형식: `C:\us valuation results\FCFF_valuation_2026-06-01_to_2026-06-02_2026-06-06.xlsx`
(= `{Valuation_method}_valuation_{수행날짜}_{출력날짜}.xlsx`, csv 는 `file_format="csv"`)

In [12]:
# ── Cell 12 · 실행 예시 (필요 시 주석 해제) ─────────────────────
# fcff_top = get_fcff_ranking(n=30)
# rim_top  = get_rim_ranking(n=30)
# rel_top  = get_relative_ranking(n=30, metrics="ALL")
# combo    = get_combined_ranking(n=50, min_models=3)

In [13]:
fcff_top = get_fcff_ranking(n=600, dates=['2026-06-30', '2026-07-01'])

[10:58:39][FCFF] ⚠️  측정 기록 없는 날짜 (무시): ['2026-07-01']
[10:59:12][GROWTH] 예측 59,440행 / 실적 85,400행 로드 (예측 보유 종목 1,868개)
[10:59:15][GROWTH] 성장률 테이블 1,868종목 (4Q 산출 1,840 / 8Q 산출 1,845) — 캐시 저장

  [FCFF DCF] upside 순위  |  평가일 ['2026-06-30']  |  TOP 600


,rank,ticker,종목명,섹터,측정일,TP($),현재가($),Upside(%),매출성장률_4Q(%),매출성장률_8Q(%),WACC(%),g_term(%),재투자율(%),ROIC평균(%),EV($B),지분가치($B),순부채($B),주식수(M),시가총액($B)
0,1,SNMP,Evolve Transition Infrastructure LP,Energy,2026-06-30,42.73,1.33,3112.5,-3.8,-33.4,9.46,0.00,10.8,-515.7,0.37,0.35,0.02,8.1,0.34
1,2,SRPT,"Sarepta Therapeutics, Inc.",Healthcare,2026-06-30,408.61,18.07,2161.0,38.3,40.2,6.63,4.00,33.3,254.0,50.39,49.82,0.57,121.9,1.90
2,3,PRGO,Perrigo Company plc,Healthcare,2026-06-30,225.30,10.46,2052.9,-1.8,-4.3,5.55,3.55,57.6,124.2,34.70,31.25,3.45,138.7,1.44
3,4,SABR,Sabre Corporation,Consumer Cyclical,2026-06-30,40.35,2.09,1830.6,-32.1,-25.8,5.55,3.55,80.4,109.3,19.64,16.04,3.60,397.6,0.83
4,5,KNOP,KNOT Offshore Partners LP,Industrials,2026-06-30,188.52,9.98,1789.0,5.5,16.0,5.55,3.55,1.0,9206.0,7.23,6.39,0.84,33.9,0.34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,596,HUBG,"Hub Group, Inc.",Industrials,2026-06-30,46.16,43.28,6.7,-1.8,-4.2,8.38,4.00,71.7,131.7,3.16,2.78,0.38,60.3,2.65
596,597,PNR,Pentair plc,Industrials,2026-06-30,81.09,76.46,6.1,3.1,4.8,8.44,4.00,9.4,745.5,15.15,13.27,1.88,163.7,12.39
597,598,MU,"Micron Technology, Inc.",Technology,2026-06-30,1212.51,1145.28,5.9,331.9,1663.5,11.03,4.00,55.9,121.2,1369.71,1388.33,-18.62,1145.0,1303.64
598,599,EFX,Equifax Inc.,Industrials,2026-06-30,164.11,155.28,5.7,9.7,18.4,8.62,4.00,81.5,122.3,24.95,19.82,5.12,120.8,19.15


[10:59:16][SAVE] 저장 완료: C:\us valuation results\FCFF_valuation_2026-06-30_2026-07-01.xlsx  (600행)


In [14]:
rim_top  = get_rim_ranking(n=600, dates=['2026-05-30'])


  [RIM] upside 순위  |  평가일 ['2026-05-30']  |  TOP 600


,rank,ticker,종목명,섹터,측정일,TP($),현재가($),Upside(%),매출성장률_4Q(%),매출성장률_8Q(%),내재가치($B),Re(%),g_term(%),RI지속계수ω,Phase2연수,Moat,시가총액($B)
0,1,MBT,Mobile TeleSystems Public Joint Stock Company,Communication Services,2026-05-30,10597999.60,5.50,192690801.8,8.3,16.2,8992301.98,14.31,13.56,0.992,30,Exceptional moat,9.28
1,2,TLK,Perusahaan Perseroan (Persero) PT Telekomunika...,Communication Services,2026-05-30,215675.73,16.42,1313394.1,3.0,3.3,213633.87,12.40,11.65,0.975,25,Wide-Narrow moat,13.28
2,3,EC,Ecopetrol S.A.,Energy,2026-05-30,87154.00,14.61,596436.6,-14.9,-6.3,179162.04,12.48,11.73,0.960,20,Narrow moat,29.28
3,4,HRB,"H&R Block, Inc.",Consumer Cyclical,2026-05-30,99695.28,38.49,258916.1,-2.4,-0.2,12909.44,11.85,11.10,0.980,28,Wide moat,4.83
4,5,PKX,POSCO Holdings Inc.,Basic Materials,2026-05-30,98734.46,70.90,139158.8,4.3,4.6,36724.48,14.92,14.17,0.620,5,No moat,15.54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,596,PKI,"PerkinElmer, Inc.",Healthcare,2026-05-30,51.85,115.24,-55.0,-7.7,-0.5,5.80,10.95,10.20,0.620,5,No moat,14.46
596,597,IQV,IQVIA Holdings Inc.,Healthcare,2026-05-30,81.76,182.21,-55.1,4.1,11.3,13.88,11.85,11.10,0.950,20,Narrow moat,32.25
597,598,SWN,Southwestern Energy Company,Energy,2026-05-30,3.19,7.11,-55.1,-4.6,-41.6,3.52,12.17,11.42,0.620,5,No moat,7.84
598,599,ROLL,RBC Bearings Incorporated,Industrials,2026-05-30,94.98,212.38,-55.3,13.6,28.9,3.01,12.51,11.76,0.620,5,No moat,6.16


[20:56:34][SAVE] 저장 완료: C:\us valuation results\RIM_valuation_2026-05-30_2026-07-01.xlsx  (600행)


In [15]:
merged = merge_upside_rankings([fcff_top, rim_top])

[20:56:42][MERGE] [FCFF] 600종목 | upside='Upside(%)' | Moat=X | 성장률=O
[20:56:42][MERGE] [RIM] 600종목 | upside='Upside(%)' | Moat=O | 성장률=O
[20:56:42][MERGE] 병합 완료: 합집합 957종목 → min_models≥1 적용 후 957종목 출력 (평균 = FCFF+RIM 동일가중)

  [Merged FCFF+RIM] 동일가중 평균 upside 순위  |  min_models=1  |  전체


,rank,ticker,종목명,섹터,최신측정일,Moat,Upside_평균(%),평가모형수,Upside_FCFF(%),Upside_RIM(%),매출성장률_4Q(%),매출성장률_8Q(%)
0,1,MBT,Mobile TeleSystems Public Joint Stock Company,Communication Services,2026-05-30,Exceptional moat,192690801.8,1,NaN,192690801.8,8.3,16.2
1,2,TLK,Perusahaan Perseroan (Persero) PT Telekomunika...,Communication Services,2026-05-30,Wide-Narrow moat,1313394.1,1,NaN,1313394.1,3.0,3.3
2,3,EC,Ecopetrol S.A.,Energy,2026-05-30,Narrow moat,596436.6,1,NaN,596436.6,-14.9,-6.3
3,4,HRB,"H&R Block, Inc.",Consumer Cyclical,2026-05-30,Wide moat,258916.1,1,NaN,258916.1,-2.4,-0.2
4,5,PKX,POSCO Holdings Inc.,Basic Materials,2026-05-30,No moat,139158.8,1,NaN,139158.8,4.3,4.6
...,...,...,...,...,...,...,...,...,...,...,...,...
952,953,PKI,"PerkinElmer, Inc.",Healthcare,2026-05-30,No moat,-55.0,1,NaN,-55.0,-7.7,-0.5
953,954,SWN,Southwestern Energy Company,Energy,2026-05-30,No moat,-55.1,1,NaN,-55.1,-4.6,-41.6
954,955,IQV,IQVIA Holdings Inc.,Healthcare,2026-05-30,Narrow moat,-55.1,1,NaN,-55.1,4.1,11.3
955,956,ROLL,RBC Bearings Incorporated,Industrials,2026-05-30,No moat,-55.3,1,NaN,-55.3,13.6,28.9


[20:56:42][SAVE] 저장 완료: C:\us valuation results\Merged_FCFF+RIM_valuation_2026-05-30_to_2026-06-30_2026-07-01.xlsx  (957행)
